# CPU parameter inspection: anchor cross attention

Executed only the setup and parameter-preview cells from the [maintained notebook](../../../../anchor_cross_attention.ipynb). This measures trainable parameter counts, not predictive performance, runtime or peak memory. Settings below are retained; only the preview switch is enabled in this copy. No dataset, cache, checkpoint or prediction was consumed. See [execution.json](execution.json) for provenance.

In [1]:
from pathlib import Path
import sys
import torch
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'Code/shared_code').is_dir())
sys.path.insert(0, str(root / 'Code'))
torch.set_num_threads(1)
torch.set_default_device('cpu')


In [2]:
# Standard-library paths, immutable configuration updates and saved JSON records.
from dataclasses import asdict, replace
import hashlib
import json
from pathlib import Path
import sys

# Data tables and plotting; import NumPy before PyTorch in the Windows environment.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from shared_code import datasets

In [3]:
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    raise FileNotFoundError("Start this notebook from within the repository.")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code import datasets
from shared_code.fusion import controlled_experiment as experiment
from shared_code.fusion import correction_experiment, run_identity, study_inputs
from shared_code.fusion import study_references, study_selection, token_embeddings, train

In [4]:
READ_SAVED_RESULTS = True
RUN_TRAINING = False
RUN_ANALYSIS = False
RUN_DIAGNOSTICS = False
RUN_PARAMETER_PREVIEW = False  # Explicit CPU construction from declared shapes; no data or weights.

# The prepared launcher selects one benchmark and one training seed per job.
BENCHMARKS = ("cov_vac_degradation", "ecoli_proteins")
BENCHMARK = None
SEED = 0
TRAINING_SEEDS = (0, 1, 2)
SELECTION_SEED = 0
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
BUNDLES = {
    "original": {"dna": "ntv2_100m", "rna": "rnafm", "prot": "esm2_8m"},
    "mrnafm": {"dna": "ntv2_100m", "rna": "mrnafm", "prot": "esm2_8m"},
}

# Retained BLF training budget; these are choices, not newly tuned settings.
BASE_CONFIG = train.TrainConfig(
    lr=3e-5, weight_decay=1e-5, batch_size=32, max_epochs=500,
    early_stopping_patience=20, plateau_patience=5,
    channels=100, tconv="equation", entropy_lambda=0.0, seed=0,
)
CONFIG = replace(BASE_CONFIG, seed=SEED)
PROJECTION_WIDTH = 600
SCORER_WIDTH = 100  # Legacy BLF setting; inactive for every attention gate in this panel.
PROJECTION_ACTIVATION = "tanh"
PROJECTION_NORM = "none"
SHARED_SCORER = False
TEMPERATURE_RANGE = (0.02, 20.0)
CONTROL_BOTTLENECK = 150  # Retained; no longer a parameter match after the joint-linear gate correction.
N_HEADS = 4
ATTENTION_DROPOUT = 0.1
UPDATE_NORM = "none"  # Nothing renormalizes the anchor after a zero auxiliary update.

DATA_ROOT = CODE_DIR.parent / "datasets"
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
REFERENCE_ROOT = CODE_DIR / "stage4" / "results" / "newton_h100_seeds012"
GROUPED_SPLIT_PATH = CODE_DIR / "stage4" / "results" / "grouped_stability_v1" / "split.json"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "anchor_cross_attention_v1"
DEVICE = "cpu"  # Reading saved records does not inspect or allocate a GPU.
PRODUCER = None  # The launcher supplies observed execution provenance, not an inferred host.

In [5]:
RUN_PARAMETER_PREVIEW = True
assert not any((RUN_TRAINING, RUN_ANALYSIS, RUN_DIAGNOSTICS))


In [6]:
if RUN_PARAMETER_PREVIEW and (RUN_TRAINING or RUN_ANALYSIS or RUN_DIAGNOSTICS):
    raise ValueError("Parameter preview is a separate CPU-only operation.")
if (PROJECTION_WIDTH, SCORER_WIDTH, PROJECTION_ACTIVATION, PROJECTION_NORM,
        SHARED_SCORER, tuple(TEMPERATURE_RANGE)) != (600, 100, "tanh", "none", False, (0.02, 20.0)):
    raise ValueError("This retained reference panel uses the declared BLF projection/scorer settings.")
if TRAINING_SEEDS != (0, 1, 2) or SELECTION_SEED != 0:
    raise ValueError("This panel fixes training seeds 0/1/2 and selection seed 0.")
if RUN_TRAINING == READ_SAVED_RESULTS:
    raise ValueError("Choose saved reading or explicit training, not both/neither.")
if RUN_TRAINING and (RUN_ANALYSIS or RUN_DIAGNOSTICS):
    raise ValueError("Analysis and diagnostic inference are separate from fitting jobs.")
if RUN_TRAINING:
    if BENCHMARK not in BENCHMARKS or SEED not in TRAINING_SEEDS:
        raise ValueError("Select one declared benchmark and training seed for this job.")
    if DEVICE != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("Training requires the verified Newton CUDA device; no CPU fallback.")
    if not isinstance(PRODUCER, dict) or not PRODUCER:
        raise ValueError("Training requires the launcher's observed producer record.")
if RUN_DIAGNOSTICS and (not isinstance(PRODUCER, dict) or not PRODUCER):
    raise ValueError("Explicit best-weight inference requires its own producer record.")
if RUN_DIAGNOSTICS and (BENCHMARK not in BENCHMARKS or SEED not in TRAINING_SEEDS):
    raise ValueError("Select one declared benchmark and seed for diagnostic inference.")
if set(BENCHMARKS) - {"cov_vac_degradation", "ecoli_proteins", "fungal_expression"}:
    raise ValueError("Grouped stability requires a separate grouped-reference adapter; do not reuse supplied-split controls.")
ACTIVE_BENCHMARKS = (BENCHMARK,) if BENCHMARK is not None else BENCHMARKS
if not set(ACTIVE_BENCHMARKS) <= set(BENCHMARKS):
    raise ValueError("The requested benchmark is outside the declared panel.")
CONFIG = replace(BASE_CONFIG, seed=SEED)
if RUN_TRAINING:
    print(f"Explicit {BENCHMARK} seed {SEED} fitting job on {DEVICE}.")
elif RUN_DIAGNOSTICS:
    print(f"Explicit {BENCHMARK} seed {SEED} best-weight diagnostic inference on {DEVICE}.")
elif RUN_ANALYSIS:
    print("Explicit CPU analysis of saved predictions and diagnostics; no fitting or inference.")
else:
    print("Saved-result reading; no fit, inference or statistical analysis is enabled.")

Saved-result reading; no fit, inference or statistical analysis is enabled.


In [7]:
PANEL = tuple({"key": variant, "variant": variant, "entropy": 0.0} for variant in (
    "anchor_only", "attention_ordinary", "attention_skip",
    "attention_sequence_sigmoid", "attention_position_sigmoid",
))
display(pd.DataFrame(PANEL))

key,variant,entropy
anchor_only,anchor_only,0.0
attention_ordinary,attention_ordinary,0.0
attention_skip,attention_skip,0.0
attention_sequence_sigmoid,attention_sequence_sigmoid,0.0
attention_position_sigmoid,attention_position_sigmoid,0.0


In [8]:
def head_settings(variant, anchor=None):
    settings = {
        "projection_width": PROJECTION_WIDTH,
        "projection_activation": PROJECTION_ACTIVATION,
        "projection_norm": PROJECTION_NORM,
    }
    if variant == "anchor_only":
        return {**settings, "anchor": anchor}
    if variant == "position_control":
        return {**settings, "control_bottleneck": CONTROL_BOTTLENECK}
    settings["scorer_width"] = SCORER_WIDTH
    if variant in experiment.SEQUENCE_VARIANTS:
        normalization = experiment.SEQUENCE_VARIANTS[variant][1]
        settings.update(shared_scorer=SHARED_SCORER,
                        temperature_range=TEMPERATURE_RANGE if normalization == "softmax" else None)
    if variant in experiment.ATTENTION_VARIANTS:
        settings.update(anchor=anchor, n_heads=N_HEADS, dropout=ATTENTION_DROPOUT,
                        update_norm=UPDATE_NORM)
    return settings


def fit_config(seed, entropy=0.0):
    return replace(BASE_CONFIG, seed=seed, entropy_lambda=entropy)


def fit_path(benchmark, bundle, key, seed):
    return RESULTS_ROOT / benchmark / bundle / key / f"seed{seed}.json"


def diagnostic_stem(benchmark, bundle, key, seed):
    return RESULTS_ROOT / benchmark / bundle / "diagnostics-v2" / key / f"seed{seed}"

In [9]:
if RUN_PARAMETER_PREVIEW:
    from types import SimpleNamespace

    # Checkpoint widths are metadata, not measurements from an opened cache.
    declared_widths = {"ntv2_100m": 512, "rnafm": 640, "mrnafm": 1280, "esm2_8m": 320}
    preview_rows = []
    for bundle, encoders in BUNDLES.items():
        widths = {modality: declared_widths[key] for modality, key in encoders.items()}
        resolutions = {modality: token_embeddings.grid_resolution(key) for modality, key in encoders.items()}
        for entry in PANEL:
            variant = entry["variant"]
            anchors = tuple(encoders) if variant == "anchor_only" or variant in experiment.ATTENTION_VARIANTS else (None,)
            for anchor in anchors:
                stream_shape = SimpleNamespace(widths=widths, resolutions=resolutions, n_outputs=1)
                with torch.random.fork_rng(devices=[]):
                    model = experiment.build_model(stream_shape, variant, fit_config(0, entry["entropy"]),
                                                   settings=head_settings(variant, anchor))
                counts = {label: sum(parameter.numel() for parameter in component.parameters() if parameter.requires_grad)
                          for label, component in (("total", model), ("fusion", model.fusion), ("prediction", model.head))}
                preview_rows.append({"bundle": bundle, "model": entry["key"], "possible anchor": anchor,
                                     "output classes": 1, **counts})
                del model
    display(pd.DataFrame(preview_rows))
    print("Regression-output preview only; classification adds 602 readout parameters for three classes. No widths were changed.")
else:
    print("Parameter construction skipped; saved result counts remain available below.")


Regression-output preview only; classification adds 602 readout parameters for three classes. No widths were changed.


bundle,model,possible anchor,output classes,total,fusion,prediction
original,anchor_only,dna,1,1553201,307800,720601
original,anchor_only,rna,1,1630001,384600,720601
original,anchor_only,prot,1,1438001,192600,720601
original,attention_ordinary,dna,1,5015201,3769800,720601
original,attention_ordinary,rna,1,5015201,3769800,720601
original,attention_ordinary,prot,1,5015201,3769800,720601
original,attention_skip,dna,1,5015201,3769800,720601
original,attention_skip,rna,1,5015201,3769800,720601
original,attention_skip,prot,1,5015201,3769800,720601
original,attention_sequence_sigmoid,dna,1,5018803,3773402,720601
